In [32]:
import pandas as pd
import numpy as np

In [33]:
# load data
file_path = "./data/colorectal_data.csv"
df = pd.read_csv(file_path)

In [34]:
df.head()

,Survival months,Vital status recode (study cutoff used),Sex,"Race recode (W, B, AI, API)",Marital status at diagnosis,Primary Site - labeled,Age recode with single ages and 90+,Grade Recode (thru 2017),Regional nodes positive (1988+),Regional nodes examined (1988+),CS tumor size (2004-2015),Tumor Size Summary (2016+),Combined Summary Stage with Expanded Regional Codes (2004+),CS site-specific factor 1 (2004-2017 varying by schema),RX Summ--Surg Prim Site (1998+),"Chemotherapy recode (yes, no/unk)",Radiation recode,CEA Pretreatment Interpretation Recode (2010+)
0,0049,Dead,Male,White,Single (never married),"C20.9-Rectum, NOS",69 years,Well differentiated; Grade I,0,10,Blank(s),Blank(s),Blank(s),Blank(s),30,No/Unknown,None/Unknown,Blank(s)
1,0179,Alive,Female,White,Single (never married),C19.9-Rectosigmoid junction,39 years,Moderately differentiated; Grade II,1,8,035,Blank(s),Distant site(s)/node(s) involved,020,30,Yes,Beam radiation,Blank(s)
2,0000,Dead,Female,White,Widowed,C18.2-Ascending colon,82 years,Poorly differentiated; Grade III,98,0,999,Blank(s),Regional by both direct extension and lymph no...,998,0,No/Unknown,None/Unknown,Blank(s)
3,0102,Alive,Female,Asian or Pacific Islander,Single (never married),"C20.9-Rectum, NOS",67 years,Moderately differentiated; Grade II,98,0,050,Blank(s),Regional by both direct extension and lymph no...,010,0,Yes,Beam radiation,CEA positive/elevated
4,0004,Dead,Male,White,Married (including common law),"C20.9-Rectum, NOS",78 years,Unknown,2,16,070,Blank(s),Regional lymph nodes involved only,998,50,No/Unknown,None/Unknown,Blank(s)


In [35]:
df.shape

(839633, 18)

In [36]:
# clean up column names
df.columns = [c.strip().replace('"', '') for c in df.columns]

In [37]:
df.head()

,Survival months,Vital status recode (study cutoff used),Sex,"Race recode (W, B, AI, API)",Marital status at diagnosis,Primary Site - labeled,Age recode with single ages and 90+,Grade Recode (thru 2017),Regional nodes positive (1988+),Regional nodes examined (1988+),CS tumor size (2004-2015),Tumor Size Summary (2016+),Combined Summary Stage with Expanded Regional Codes (2004+),CS site-specific factor 1 (2004-2017 varying by schema),RX Summ--Surg Prim Site (1998+),"Chemotherapy recode (yes, no/unk)",Radiation recode,CEA Pretreatment Interpretation Recode (2010+)
0,0049,Dead,Male,White,Single (never married),"C20.9-Rectum, NOS",69 years,Well differentiated; Grade I,0,10,Blank(s),Blank(s),Blank(s),Blank(s),30,No/Unknown,None/Unknown,Blank(s)
1,0179,Alive,Female,White,Single (never married),C19.9-Rectosigmoid junction,39 years,Moderately differentiated; Grade II,1,8,035,Blank(s),Distant site(s)/node(s) involved,020,30,Yes,Beam radiation,Blank(s)
2,0000,Dead,Female,White,Widowed,C18.2-Ascending colon,82 years,Poorly differentiated; Grade III,98,0,999,Blank(s),Regional by both direct extension and lymph no...,998,0,No/Unknown,None/Unknown,Blank(s)
3,0102,Alive,Female,Asian or Pacific Islander,Single (never married),"C20.9-Rectum, NOS",67 years,Moderately differentiated; Grade II,98,0,050,Blank(s),Regional by both direct extension and lymph no...,010,0,Yes,Beam radiation,CEA positive/elevated
4,0004,Dead,Male,White,Married (including common law),"C20.9-Rectum, NOS",78 years,Unknown,2,16,070,Blank(s),Regional lymph nodes involved only,998,50,No/Unknown,None/Unknown,Blank(s)


In [38]:
df["Regional nodes examined (1988+)"].unique()

array([10,  8,  0, 16,  7, 14, 13, 30, 18, 20, 98, 27, 66, 23,  9, 12, 53,
       99, 11, 34,  6, 17, 15, 19,  5,  3,  4, 29, 33, 44, 24,  2, 36,  1,
       32, 41, 21, 22, 26, 35, 80, 25, 58, 48, 40, 38, 31, 69, 52, 43, 39,
       28, 55, 54, 97, 90, 42, 67, 49, 56, 46, 37, 84, 45, 95, 70, 50, 47,
       59, 64, 79, 51, 96, 73, 61, 72, 65, 57, 62, 83, 63, 78, 82, 71, 68,
       74, 60, 89, 77, 86, 81, 85, 76, 88, 87, 75])

In [39]:
# SEER uses codes for certain variables, we convert them to N/A
cleaning_rules = {
    "Survival months": [999],
    "Age recode with single ages and 90+": [999],
    "Regional nodes positive (1988+)": [99, 98], # 99=Unknown, 98=90+
    "Regional nodes examined (1988+)": [99, 98, 97, 96], # Unknown/not examined
    "CS tumor size (2004-2015)": range(990, 1000), # 990-999 are all unknown/diffuse
    "Tumor Size Summary (2016+)": range(990, 1000),
    "CS site-specific factor 1 (2004-2017 varying by schema)": [998, 999],
    # 7 = Test done but exact value unknown, 9 = Unknown if test was done
    "CEA Pretreatment Interpretation Recode (2010+)": [7, 9, 998, 999],
    "RX Summ--Surg Prim Site (1998+)": [99]
}

def cleaner(col_name, value):
    # Convert to string to handle 'Labels' vs 'Values'
    val_str = str(value).strip()

    # universal string unknowns
    universal_unknowns = ['Unknown', 'Blank(s)', 'Unknown/not reached', 'Not applicable', 'NOS']
    if any(unk in val_str for unk in universal_unknowns):
        return np.nan

    # column-specific numeric trash codes
    try:
        num_val = float(val_str)
        if col_name in cleaning_rules:
            if num_val in cleaning_rules[col_name]:
                return np.nan
        return num_val
    except ValueError:
        return val_str

# 3. Apply the cleaner to EVERY column
for col in df.columns:
    df[col] = df[col].apply(lambda x: cleaner(col, x))

In [40]:
df.head()

,Survival months,Vital status recode (study cutoff used),Sex,"Race recode (W, B, AI, API)",Marital status at diagnosis,Primary Site - labeled,Age recode with single ages and 90+,Grade Recode (thru 2017),Regional nodes positive (1988+),Regional nodes examined (1988+),CS tumor size (2004-2015),Tumor Size Summary (2016+),Combined Summary Stage with Expanded Regional Codes (2004+),CS site-specific factor 1 (2004-2017 varying by schema),RX Summ--Surg Prim Site (1998+),"Chemotherapy recode (yes, no/unk)",Radiation recode,CEA Pretreatment Interpretation Recode (2010+)
0,49.0,Dead,Male,White,Single (never married),NaN,69 years,Well differentiated; Grade I,0.0,10.0,NaN,NaN,NaN,NaN,30.0,NaN,NaN,NaN
1,179.0,Alive,Female,White,Single (never married),C19.9-Rectosigmoid junction,39 years,Moderately differentiated; Grade II,1.0,8.0,35.0,NaN,Distant site(s)/node(s) involved,20.0,30.0,Yes,Beam radiation,NaN
2,0.0,Dead,Female,White,Widowed,C18.2-Ascending colon,82 years,Poorly differentiated; Grade III,NaN,0.0,NaN,NaN,Regional by both direct extension and lymph no...,NaN,0.0,NaN,NaN,NaN
3,102.0,Alive,Female,Asian or Pacific Islander,Single (never married),NaN,67 years,Moderately differentiated; Grade II,NaN,0.0,50.0,NaN,Regional by both direct extension and lymph no...,10.0,0.0,Yes,Beam radiation,CEA positive/elevated
4,4.0,Dead,Male,White,Married (including common law),NaN,78 years,NaN,2.0,16.0,70.0,NaN,Regional lymph nodes involved only,NaN,50.0,NaN,NaN,NaN


In [41]:
# merge overlapping columns
# combine tumor sizes (2004 - 2015 and 2016+)
df["Tumor_Size"] = df["Tumor Size Summary (2016+)"].fillna(df["CS tumor size (2004-2015)"])

# combine CEA biomarker
df["CEA_Result"] = df["CEA Pretreatment Interpretation Recode (2010+)"].fillna(df["CS site-specific factor 1 (2004-2017 varying by schema)"])

In [42]:
# targets
df["T"] = pd.to_numeric(df["Survival months"], errors='coerce')
df["E"] = df["Vital status recode (study cutoff used)"].apply(
    lambda x: 1 if "Dead" in str(x) else 0
)

In [43]:
# drop old raw columns
to_drop = [
    "Survival months", "Vital status recode (study cutoff used)",
    "CS tumor size (2004-2015)", "Tumor Size Summary (2016+)",
    "CS site-specific factor 1 (2004-2017 varying by schema)",
    "CEA Pretreatment Interpretation Recode (2010+)"
]

df_final = df.drop(columns=to_drop)

In [44]:
# drop rows without survival time or event
df_final = df_final.dropna(subset=["T", "E"])

In [45]:
df_final.shape

(831287, 16)

In [46]:
df_final.head()

,Sex,"Race recode (W, B, AI, API)",Marital status at diagnosis,Primary Site - labeled,Age recode with single ages and 90+,Grade Recode (thru 2017),Regional nodes positive (1988+),Regional nodes examined (1988+),Combined Summary Stage with Expanded Regional Codes (2004+),RX Summ--Surg Prim Site (1998+),"Chemotherapy recode (yes, no/unk)",Radiation recode,Tumor_Size,CEA_Result,T,E
0,Male,White,Single (never married),NaN,69 years,Well differentiated; Grade I,0.0,10.0,NaN,30.0,NaN,NaN,NaN,NaN,49.0,1
1,Female,White,Single (never married),C19.9-Rectosigmoid junction,39 years,Moderately differentiated; Grade II,1.0,8.0,Distant site(s)/node(s) involved,30.0,Yes,Beam radiation,35.0,20.0,179.0,0
2,Female,White,Widowed,C18.2-Ascending colon,82 years,Poorly differentiated; Grade III,NaN,0.0,Regional by both direct extension and lymph no...,0.0,NaN,NaN,NaN,NaN,0.0,1
3,Female,Asian or Pacific Islander,Single (never married),NaN,67 years,Moderately differentiated; Grade II,NaN,0.0,Regional by both direct extension and lymph no...,0.0,Yes,Beam radiation,50.0,CEA positive/elevated,102.0,0
4,Male,White,Married (including common law),NaN,78 years,NaN,2.0,16.0,Regional lymph nodes involved only,50.0,NaN,NaN,70.0,NaN,4.0,1


In [47]:
# convert age column to numeric
df_final["Age recode with single ages and 90+"] = (
    df_final["Age recode with single ages and 90+"]
    .astype(str)
    .str.replace("90+ years", "90", case=False, regex=False)
    .str.replace(" years", "", case=False)
)
df_final["Age recode with single ages and 90+"] = pd.to_numeric(df_final["Age recode with single ages and 90+"])

In [48]:
df_final.head()

,Sex,"Race recode (W, B, AI, API)",Marital status at diagnosis,Primary Site - labeled,Age recode with single ages and 90+,Grade Recode (thru 2017),Regional nodes positive (1988+),Regional nodes examined (1988+),Combined Summary Stage with Expanded Regional Codes (2004+),RX Summ--Surg Prim Site (1998+),"Chemotherapy recode (yes, no/unk)",Radiation recode,Tumor_Size,CEA_Result,T,E
0,Male,White,Single (never married),NaN,69,Well differentiated; Grade I,0.0,10.0,NaN,30.0,NaN,NaN,NaN,NaN,49.0,1
1,Female,White,Single (never married),C19.9-Rectosigmoid junction,39,Moderately differentiated; Grade II,1.0,8.0,Distant site(s)/node(s) involved,30.0,Yes,Beam radiation,35.0,20.0,179.0,0
2,Female,White,Widowed,C18.2-Ascending colon,82,Poorly differentiated; Grade III,NaN,0.0,Regional by both direct extension and lymph no...,0.0,NaN,NaN,NaN,NaN,0.0,1
3,Female,Asian or Pacific Islander,Single (never married),NaN,67,Moderately differentiated; Grade II,NaN,0.0,Regional by both direct extension and lymph no...,0.0,Yes,Beam radiation,50.0,CEA positive/elevated,102.0,0
4,Male,White,Married (including common law),NaN,78,NaN,2.0,16.0,Regional lymph nodes involved only,50.0,NaN,NaN,70.0,NaN,4.0,1


In [49]:
# fill missing values in numeric columns with median
numeric_cols = ["Age recode with single ages and 90+", "Regional nodes positive (1988+)",
                "Regional nodes examined (1988+)", "Tumor_Size", "CEA_Result"]

for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
    median_val = df_final[col].median()
    df_final[col] = df_final[col].fillna(median_val)

In [50]:
# ordinal encoding for grade and stage
grade_map = {
    "Well differentiated; Grade I": 1,
    "Moderately differentiated; Grade II": 2,
    "Poorly differentiated; Grade III": 3,
    "Undifferentiated; anaplastic; Grade IV": 4,
}

stage_map = {
    "In situ": 0,
    "Localized only": 1,
    "Regional lymph nodes involved only": 2,
    "Regional by both direct extension and lymph nodes": 3,
    "Distant site(s)/node(s) involved": 4,
}
df_final["Grade_Ordinal"] = df_final["Grade Recode (thru 2017)"].map(grade_map).fillna(0)
df_final["Stage_Ordinal"] = df_final["Combined Summary Stage with Expanded Regional Codes (2004+)"].map(stage_map).fillna(0)

In [51]:
# grouping anatomy sites into broader categories
def group_anatomy(site):
    site = str(site).lower()
    if site == "nan":
        return "Unknown"
    if any(x in site for x in ["rectum", "rectosigmoid", "rectal"]):
        return "Rectum"
    elif any(x in site for x in ["cecum", "ascending", "hepatic flexure", "transverse", "appendix"]):
        return "Right_Colon"
    elif any(x in site for x in ["splenic flexure", "descending", "sigmoid"]):
        return "Left_Colon"
    else:
        return "Other_Unknown"

df_final["Anatomy_Group"] = df_final["Primary Site - labeled"].apply(group_anatomy)

In [52]:
# create a binary feature for marital status (Married vs Not Married)
df_final["Is_Married"] = df_final["Marital status at diagnosis"].apply(
    lambda x: 1 if "Married" in str(x) else 0
)

In [53]:
# group surgical codes into broader categories
def group_surgery(code):
    # Handle missing/unknowns first
    code_str = str(code).strip().lower()
    if code_str in ["nan", "unknown", "99.0", "99"]:
        return "Unknown"

    try:
        val = float(code)
        if val == 0:
            return "No_Surgery"
        elif 10 <= val <= 29:
            # Polypectomies and local excisions
            return "Local_Excision"
        elif 30 <= val <= 49:
            # Removing part of the colon (e.g., right hemicolectomy)
            return "Partial_Colectomy"
        elif 50 <= val <= 79:
            # Removing the entire colon or rectum
            return "Total_Colectomy"
        elif val >= 80:
            # Radical surgeries bypassing standard boundaries or Not Otherwise Specified
            return "Radical_or_NOS"
        else:
            return "Unknown"
    except ValueError:
        return "Unknown"

# apply grouping
df_final["Surgery_Group"] = df_final["RX Summ--Surg Prim Site (1998+)"].apply(group_surgery)

# drop initial column
df_final = df_final.drop(columns=["RX Summ--Surg Prim Site (1998+)"])

In [54]:
# one hot encode categorical variables
categorical_cols = [
    "Sex",
    "Race recode (W, B, AI, API)",
    "Chemotherapy recode (yes, no/unk)",
    "Radiation recode",
    "Anatomy_Group",
    #"RX Summ--Surg Prim Site (1998+)",
    "Surgery_Group",
    "CEA_Result"
]

# redo the categorical cleaning to ensure all trash codes are converted to 'Unknown' before encoding
for col in categorical_cols:
    df_final[col] = df_final[col].fillna("Unknown")

    trash_codes = [
        997, 997.0, "997", "997.0",
        998, 998.0, "998", "998.0",
        999, 999.0, "999", "999.0"
    ]
    df_final[col] = df_final[col].replace(trash_codes, "Unknown")
    df_final[col] = df_final[col].astype(str)

In [55]:
cols_to_drop = [
    "Grade Recode (thru 2017)",
    "Combined Summary Stage with Expanded Regional Codes (2004+)",
    "Primary Site - labeled",
    "Marital status at diagnosis"
]
df_encoded = df_final.drop(columns=cols_to_drop)

In [56]:
df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True, dtype="float32")

In [57]:
df_encoded.head()

,Age recode with single ages and 90+,Regional nodes positive (1988+),Regional nodes examined (1988+),Tumor_Size,T,E,Grade_Ordinal,Stage_Ordinal,Is_Married,Sex_Male,...,Anatomy_Group_Right_Colon,Anatomy_Group_Unknown,Surgery_Group_No_Surgery,Surgery_Group_Partial_Colectomy,Surgery_Group_Radical_or_NOS,Surgery_Group_Total_Colectomy,Surgery_Group_Unknown,CEA_Result_20.0,CEA_Result_30.0,CEA_Result_Unknown
0,69,0.0,10.0,40.0,49.0,1,1.0,0.0,0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,39,1.0,8.0,35.0,179.0,0,2.0,4.0,0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,82,0.0,0.0,40.0,0.0,1,3.0,0.0,0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,67,0.0,0.0,50.0,102.0,0,2.0,0.0,0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,78,2.0,16.0,70.0,4.0,1,0.0,2.0,1,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [58]:
df_encoded.columns

Index(['Age recode with single ages and 90+',
       'Regional nodes positive (1988+)', 'Regional nodes examined (1988+)',
       'Tumor_Size', 'T', 'E', 'Grade_Ordinal', 'Stage_Ordinal', 'Is_Married',
       'Sex_Male', 'Race recode (W, B, AI, API)_Asian or Pacific Islander',
       'Race recode (W, B, AI, API)_Black',
       'Race recode (W, B, AI, API)_Unknown',
       'Race recode (W, B, AI, API)_White',
       'Chemotherapy recode (yes, no/unk)_Yes',
       'Radiation recode_Combination of beam with implants or isotopes',
       'Radiation recode_Radioactive implants (includes brachytherapy) (1988+)',
       'Radiation recode_Radioisotopes (1988+)',
       'Radiation recode_Recommended, unknown if administered',
       'Radiation recode_Refused (1988+)', 'Radiation recode_Unknown',
       'Anatomy_Group_Other_Unknown', 'Anatomy_Group_Rectum',
       'Anatomy_Group_Right_Colon', 'Anatomy_Group_Unknown',
       'Surgery_Group_No_Surgery', 'Surgery_Group_Partial_Colectomy',
       'S

In [59]:
df_encoded.to_csv("./data/colorectal_data_cleaned.csv", index=False)